In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L

NUM_CLASSES = 9  # adjust if your label space changes

class RFICNN(L.LightningModule):
    def __init__(self, num_classes=NUM_CLASSES, lr=1e-3, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters()
        # Simple 2D CNN for spectrogram-like inputs
        self.net = nn.Sequential(
            # input: (B, 1, H, W)
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(64, num_classes)
        self.lr = lr
        self.weight_decay = weight_decay
        self.train_acc = L.metrics.classification.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = L.metrics.classification.Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc = L.metrics.classification.Accuracy(task="multiclass", num_classes=num_classes)
        self.criterion = nn.CrossEntropyLoss()
    
    # preparing the model 
    def forward(self, x):
        # x is (B, H, W); add channel dim for CNN
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.net(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = self.train_acc(preds, y)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = self.val_acc(preds, y)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = self.test_acc(preds, y)
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}
        }

val_loader = test_loader

model = RFICNN(num_classes=NUM_CLASSES, lr=1e-3, weight_decay=1e-4)

trainer = L.Trainer(
    max_epochs=15,
    accelerator="auto",
    devices="auto",
    log_every_n_steps=10
)

# Train and validate
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

# Final test on held-out data
trainer.test(model, dataloaders=test_loader)

NameError: name 'full_ds' is not defined